# 01 – Fruktbarhet i riket

**Syfte:** Göra om SCB:s rikssiffror för summerad fruktsamhet 2000–2025 till en ren referenstabell med en rad per år.

Tabellen används i `02_FertilitetPerKommun` för att **avtrenda** kommunernas fruktsamhet. Varje kommun jämförs då med riket samma år. På så vis försvinner den nationella trenden, och reglerna i ARM hittar inte bara vilket år det är.

| | |
|---|---|
| **Källa** | SCB, Statistikdatabasen: summerad fruktsamhet för riket, per kön |
| **Rådata** | `data/raw/Fruktbarhet_riket.csv` (oförändrad, identisk med första versionen i git, commit `2413c0a`) |
| **Resultat** | `data/processed/fruktbarhet_riket.csv` |
| **Beroenden** | Inga. Körs **först**. |

**Körordning för hela mappen:** `01` → `02` → `03`. `04` är fristående.

*Summerad fruktsamhet* är det antal barn en kvinna (eller man) skulle få under livet om årets fruktsamhet i varje ålder gällde hela livet. Talet är ett mått per person, inte ett antal födda.

Den tidigare filen `Fruktbarhet_riket_kvinnor.csv` ingår inte. Enligt `Datagranskning_NY.md` innehöll den samma kvinnoserie som den här filen, och den har redan tagits bort ur `CSV-files`.

## 1. Inställningar

Sökvägar till rådata och till mappen för bearbetade filer. Cellen fungerar oavsett om notebooken körs från `Assignment` eller från projektroten.

In [1]:
from pathlib import Path

import pandas as pd

# Notebooken körs normalt från mappen Assignment.
# Körs den från projektroten letar vi upp Assignment ändå.
BAS = Path.cwd()
if not (BAS / "data" / "raw").is_dir() and (BAS / "Assignment").is_dir():
    BAS = BAS / "Assignment"

RAW = BAS / "data" / "raw"
PROCESSED = BAS / "data" / "processed"
PROCESSED.mkdir(parents=True, exist_ok=True)

assert RAW.is_dir(), "Hittar inte data/raw. Öppna notebooken från mappen Assignment."

## 2. Läs in rådata

Filen är kommaseparerad och kodad i UTF-8 med BOM. Därför används `encoding="utf-8-sig"`, annars får första kolumnnamnet ett osynligt tecken framför sig. Varje rad är ett kön och ett år. Värdekolumnen heter `Summerad fruktsamhet, antal`. Kommatecknet i namnet står inom citattecken i filen, så det delar inte kolumnen.

In [2]:
riket_ra = pd.read_csv(RAW / "Fruktbarhet_riket.csv", encoding="utf-8-sig")

display(riket_ra.head())
print("Form:", riket_ra.shape)
print(riket_ra.dtypes)

,region,kön,år,"Summerad fruktsamhet, antal"
0,00 Riket,män,2000,1.42
1,00 Riket,män,2001,1.44
2,00 Riket,män,2002,1.52
3,00 Riket,män,2003,1.57
4,00 Riket,män,2004,1.61


Form: (52, 4)
region                             str
kön                                str
år                               int64
Summerad fruktsamhet, antal    float64
dtype: object


## 3. Kontrollera strukturen

Vi förväntar oss **2 kön × 26 år = 52 rader**: en enda region, inga dubbletter och inga saknade värden. Varje antagande prövas med `assert`. Om en ny nedladdning från SCB ser annorlunda ut stannar notebooken här, i stället för att ett fel smyger sig vidare.

In [3]:
VARDE = "Summerad fruktsamhet, antal"

assert list(riket_ra.columns) == ["region", "kön", "år", VARDE]
assert riket_ra["region"].nunique() == 1
assert set(riket_ra["kön"]) == {"män", "kvinnor"}
assert riket_ra["år"].min() == 2000 and riket_ra["år"].max() == 2025
assert len(riket_ra) == 2 * 26
assert not riket_ra.duplicated(["kön", "år"]).any()
assert riket_ra[VARDE].notna().all()

print("Region:", riket_ra["region"].iloc[0])
print("År:", riket_ra["år"].min(), "–", riket_ra["år"].max())
print("Värden:", riket_ra[VARDE].min(), "–", riket_ra[VARDE].max())

Region: 00 Riket
År: 2000 – 2025
Värden: 1.29 – 1.98


## 4. Rensa och ändra till bred form

- `region` är samma på alla rader (`00 Riket`) och tas bort.
- Kolumnnamnen görs om till ASCII-tecken (`år` → `ar`, `män` → `man`), så att de fungerar likadant i alla miljöer.
- Tabellen får **bred form**: en rad per år och en kolumn per kön. Då kan varje kommun-år i `02` kopplas till rätt rikstal med en enkel sammankoppling på `ar`.

In [4]:
riket = (
    riket_ra
    .pivot(index="år", columns="kön", values=VARDE)
    .rename(columns={"kvinnor": "fruktsamhet_riket_kvinnor", "män": "fruktsamhet_riket_man"})
    .rename_axis(index="ar", columns=None)
    .reset_index()
    [["ar", "fruktsamhet_riket_kvinnor", "fruktsamhet_riket_man"]]
)

display(riket.head())
print("Form:", riket.shape)

,ar,fruktsamhet_riket_kvinnor,fruktsamhet_riket_man
0,2000,1.54,1.42
1,2001,1.57,1.44
2,2002,1.65,1.52
3,2003,1.71,1.57
4,2004,1.75,1.61


Form: (26, 3)


## 5. Rimlighetskontroll

Två enkla kontroller:

- Alla år 2000–2025 finns, och varje år förekommer en gång.
- Kvinnornas tal är högre än männens alla år i den här serien. Kontrollen säkerställer att kolumnerna inte har bytt plats vid omformningen.

Tabellen visar också den nationella trenden. Fruktsamheten steg till omkring 2010 och har sjunkit sedan dess. Det är just den trenden som avtrendningen i `02` tar bort.

In [5]:
assert riket["ar"].tolist() == list(range(2000, 2026))
assert (riket["fruktsamhet_riket_kvinnor"] > riket["fruktsamhet_riket_man"]).all()

display(riket.loc[riket["ar"].isin([2000, 2005, 2010, 2015, 2020, 2025])])

,ar,fruktsamhet_riket_kvinnor,fruktsamhet_riket_man
0,2000,1.54,1.42
5,2005,1.77,1.62
10,2010,1.98,1.82
15,2015,1.85,1.71
20,2020,1.66,1.52
25,2025,1.42,1.29


## 6. Spara

Resultatet sparas med `utf-8-sig`, så att å, ä och ö visas rätt även när filen öppnas i Excel.

In [6]:
utfil = PROCESSED / "fruktbarhet_riket.csv"
riket.to_csv(utfil, index=False, encoding="utf-8-sig")

print(f"Sparade {len(riket)} rader till {utfil.relative_to(BAS)}")

Sparade 26 rader till data/processed/fruktbarhet_riket.csv
